In [0]:
#### Loading libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Question1: 

### Level:

**Basic**

### Question:

You are given an employee dataset containing employee details.

**Task:**
Using PySpark, select only the following columns:

* `employee_id`
* `employee_name`
* `department`
* `salary`

Then display the resulting DataFrame.

### Practical Use Case:

In a real-world ETL pipeline, source systems often contain many columns, while downstream processes require only a subset. Selecting only the required columns is a common **data transformation and column-pruning** step.

### PySpark Query for Table Creation:

```python
data = [
    (101, "Rahul", "IT", 65000, "Delhi"),
    (102, "Priya", "HR", 55000, "Mumbai"),
    (103, "Amit", "Finance", 72000, "Pune"),
    (104, "Sneha", "IT", 68000, "Bangalore"),
    (105, "Arjun", "Sales", 50000, "Hyderabad")
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary",
    "city"
]

df = spark.createDataFrame(data, columns)

df.display()
```

### Expected Output:

| employee_id | employee_name | department | salary |
| ----------: | ------------- | ---------- | -----: |
|         101 | Rahul         | IT         |  65000 |
|         102 | Priya         | HR         |  55000 |
|         103 | Amit          | Finance    |  72000 |
|         104 | Sneha         | IT         |  68000 |
|         105 | Arjun         | Sales      |  50000 |

**Your task:** Write the PySpark transformation that produces this output.

In [0]:
## Question 1:
data = [
    (101, "Rahul", "IT", 65000, "Delhi"),
    (102, "Priya", "HR", 55000, "Mumbai"),
    (103, "Amit", "Finance", 72000, "Pune"),
    (104, "Sneha", "IT", 68000, "Bangalore"),
    (105, "Arjun", "Sales", 50000, "Hyderabad")
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary",
    "city"
]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
## Answer 1:
df.select(col("employee_id"), col("employee_name"), col("department"), col("salary")).display()

**Question 1**

**Level:**
Basic

**Question:**
You have just ingested a raw file containing employee records into a PySpark DataFrame. Before writing this data to our curated tables, you need to standardize the schema.
Write a PySpark query to select only the `emp_id`, `full_name`, and `department` columns. While selecting them, rename the `full_name` column to `employee_name` to match our target database schema.

**Practical Use Case:**
In a real-world ETL pipeline, raw data (Bronze layer) often arrives with inconsistent or messy column names from various source systems (like APIs or CSVs). Renaming columns and selecting only the required fields is one of the most fundamental first steps when moving data to a standardized Silver layer.

**PySpark Query for Table Creation:**

```python
data = [
    (1001, "Alice Smith", "Data Engineering", 110000, "2021-06-15"),
    (1002, "Bob Jones", "Marketing", 75000, "2022-03-10"),
    (1003, "Charlie Brown", "Sales", 85000, "2020-11-25"),
    (1004, "Diana Prince", "Data Engineering", 125000, "2019-08-01")
]

columns = ["emp_id", "full_name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| emp_id | employee_name | department |
| --- | --- | --- |
| 1001 | Alice Smith | Data Engineering |
| 1002 | Bob Jones | Marketing |
| 1003 | Charlie Brown | Sales |
| 1004 | Diana Prince | Data Engineering |


In [0]:
## Question 1: 
data = [
    (1001, "Alice Smith", "Data Engineering", 110000, "2021-06-15"),
    (1002, "Bob Jones", "Marketing", 75000, "2022-03-10"),
    (1003, "Charlie Brown", "Sales", 85000, "2020-11-25"),
    (1004, "Diana Prince", "Data Engineering", 125000, "2019-08-01")
]

columns = ["emp_id", "full_name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer 1:
df.select(col("emp_id"), col("full_name").alias("employee_name"), col("department")).display()

### Question 2

**Level:**
Basic

**Question:**
You are building a Data Quality (DQ) check for the next step of the pipeline. Write a PySpark query to **filter out (remove)** any records where the `department` is NULL OR the `salary` is less than 50,000.

**Practical Use Case:**
In a real-world Data Engineering pipeline, filtering out "bad" or incomplete data is a critical step when moving data from the Bronze (raw) layer to the Silver (cleansed) layer. You must ensure that downstream analytics dashboards do not break due to missing dimensions (like a NULL department) or invalid metrics (like negative/abnormally low salaries).

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", None, 95000),          # Invalid: NULL department
    (3, "Charlie", "Sales", 45000),   # Invalid: Salary < 50000
    (4, "Diana", "Marketing", 85000),
    (5, "Eve", "Engineering", -5000)  # Invalid: Salary < 50000
]

columns = ["emp_id", "employee_name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| emp_id | employee_name | department | salary |
| --- | --- | --- | --- |
| 1 | Alice | Engineering | 120000 |
| 4 | Diana | Marketing | 85000 |

In [0]:
## Question:
data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", None, 95000),          # Invalid: NULL department
    (3, "Charlie", "Sales", 45000),   # Invalid: Salary < 50000
    (4, "Diana", "Marketing", 85000),
    (5, "Eve", "Engineering", -5000)  # Invalid: Salary < 50000
]

columns = ["emp_id", "employee_name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.filter((col("department").isNotNull()) & (col("salary") >= 50000)).display()

### Question 3

**Level:**
Basic

**Question:**
You are processing a batch of daily transactions. Due to an upstream system glitch, some transactions were sent multiple times, resulting in duplicate rows.
Write a PySpark query to remove these duplicates, ensuring that each `transaction_id` appears only once in the final DataFrame.

**Practical Use Case:**
Idempotency and deduplication are foundational to ETL. Upstream source systems (like Kafka, REST APIs, or transactional databases) frequently resend data due to network timeouts or retries. A robust Data Engineering pipeline must gracefully handle duplicate data to avoid double-counting revenue or inflating metrics.

**PySpark Query for Table Creation:**

```python
data = [
    ("TXN-101", "user_1", 150.00, "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"),
    ("TXN-101", "user_1", 150.00, "2023-10-01"), # Exact duplicate
    ("TXN-103", "user_3", 75.25,  "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"), # Exact duplicate
    ("TXN-104", "user_1", 50.00,  "2023-10-01")
]

columns = ["transaction_id", "user_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**
*(Note: The order of the output rows does not matter, as Spark processes data in a distributed manner.)*

| transaction_id | user_id | amount | transaction_date |
| --- | --- | --- | --- |
| TXN-101 | user_1 | 150.00 | 2023-10-01 |
| TXN-102 | user_2 | 200.50 | 2023-10-01 |
| TXN-103 | user_3 | 75.25 | 2023-10-01 |
| TXN-104 | user_1 | 50.00 | 2023-10-01 |

In [0]:
## Question:
data = [
    ("TXN-101", "user_1", 150.00, "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"),
    ("TXN-101", "user_1", 150.00, "2023-10-01"), # Exact duplicate
    ("TXN-103", "user_3", 75.25,  "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"), # Exact duplicate
    ("TXN-104", "user_1", 50.00,  "2023-10-01")
]

columns = ["transaction_id", "user_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.dropDuplicates(subset=["transaction_id"]).display()